# Exercise: Non-Graphical EDA — Swiss EV Charging Stations

In this exercise you will apply the non-graphical EDA methods from the concept notebook to a new dataset: **Swiss public EV charging stations** (source: [Swiss Federal Office of Energy, OICP open data](https://opendata.swiss/en/dataset/ladestellen-fuer-elektroautos)).

## Dataset description
| Column | Description |
|---|---|
| `evse_id` | Unique station identifier |
| `station_name` | Name of the charging station |
| `street`, `city`, `postal_code` | Location |
| `lat`, `lon` | GPS coordinates |
| `power_kw` | Maximum charging power in kilowatts |
| `power_type` | AC_3_PHASE / DC / AC_1_PHASE |
| `amperage` | Current in amperes |
| `voltage` | Voltage in volts |
| `plug_type` | Connector type (Type 2, CCS, CHAdeMO, …) |
| `is_open_24h` | Open 24 hours? |
| `renewable_energy` | Powered by renewables? |
| `auth_mode` | Authentication method |

---
## Tasks
1. Load the dataset and inspect its shape, data types, missing values and duplicates.
2. Compute mean, median and mode for `power_kw`.
3. Compute the 5th, 25th, 50th, 75th and 95th percentile for `power_kw`, `amperage` and `voltage`.
4. Compute variance and standard deviation for the numerical columns.
5. Compute skewness and kurtosis for `power_kw`. Interpret the results.
6. Build a contingency table of `power_type` vs. `plug_type`.
7. Compute a Pearson correlation matrix for the numerical columns and interpret the strongest correlation.

In [3]:
# Libraries
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

### Task 1 — Load and inspect the dataset

In [4]:
df = pd.read_csv('ev_charging_stations.csv')                #CSV-Datei einlesen und im DataFrame df speichern
                                                            #../data/: Eine Verzeichnisebene nach oben, dann in den Ordner data wechseln
                                                            #pd.read_csv(): CSV-Datei mit pandas einlesen

print('Shape:', df.shape)                                   #Anzahl Zeilen und Spalten ausgeben
                                                            #df.shape: Gibt ein Tupel (Zeilen, Spalten) zurück

df.head(3)                                                  #Die ersten drei Zeilen des DataFrames anzeigen
                                                            #head(3): 3 bestimmt die Anzahl angezeigter Zeilen

Shape: (2442, 15)


,evse_id,station_name,street,city,postal_code,lat,lon,power_kw,power_type,amperage,voltage,plug_type,is_open_24h,renewable_energy,auth_mode
0,CH*CCI*E22078,SIG CERN Esplanade des Particules,SIG CERN Esplanade des Particules,Meyrin,1217,46.23432,6.055602,22.0,AC_3_PHASE,32,230,Type 2 Outlet,True,False,NFC RFID Classic
1,CH*CCI*E22076,SIG CERN Esplanade des Particules,SIG CERN Esplanade des Particules,Meyrin,1217,46.23432,6.055602,22.0,AC_3_PHASE,32,230,Type 2 Outlet,True,False,NFC RFID Classic
2,CH*CCI*E22079,SIG CERN Esplanade des Particules,SIG CERN Esplanade des Particules,Meyrin,1217,46.23432,6.055602,22.0,AC_3_PHASE,32,230,Type 2 Outlet,True,False,NFC RFID Classic


In [5]:
# Shape, data types, missing values, duplicates
print('Data types:')                                        #Überschrift für die Datentypen ausgeben
print(df.dtypes)                                            #Datentyp jeder Spalte anzeigen (z.B. int64, float64, object)
                                                            #dtypes: Gibt den Datentyp jeder Spalte zurück
print()                                                     #Leerzeile zur besseren Lesbarkeit

print('Missing values per column:')                         #Überschrift für fehlende Werte ausgeben
print(df.isnull().sum())                                    #Anzahl fehlender Werte pro Spalte zählen
                                                            #isnull(): Prüft jede Zelle auf fehlende Werte (True = fehlend, False = vorhanden)
                                                            #sum(): Addiert die True-Werte pro Spalte (True = 1, False = 0)
print()                                                     #Leerzeile zur besseren Lesbarkeit

print(f'Duplicate rows: {df.duplicated().sum()}')           #Anzahl vollständig duplizierter Zeilen ausgeben
                                                            #duplicated(): Markiert doppelte Zeilen mit True (erstes Vorkommen = False)
                                                            #sum(): Zählt die als True markierten Duplikate
                                                            #f'...': Fügt das berechnete Ergebnis innerhalb von {} in den Text ein

Data types:
evse_id                 str
station_name            str
street                  str
city                    str
postal_code             str
lat                 float64
lon                 float64
power_kw            float64
power_type              str
amperage              int64
voltage               int64
plug_type               str
is_open_24h            bool
renewable_energy       bool
auth_mode               str
dtype: object

Missing values per column:
evse_id             0
station_name        0
street              0
city                0
postal_code         0
lat                 0
lon                 0
power_kw            0
power_type          0
amperage            0
voltage             0
plug_type           0
is_open_24h         0
renewable_energy    0
auth_mode           0
dtype: int64

Duplicate rows: 0


### Task 2 — Mean, median and mode of `power_kw`

In [6]:
print(f"Mean:   {df['power_kw'].mean():.2f} kW")                                #Arithmetischen Mittelwert der Ladeleistung berechnen
                                                                                #mean(): Summe aller Werte geteilt durch deren Anzahl
                                                                                #:.2f: Ergebnis als Dezimalzahl mit zwei Nachkommastellen formatieren

print(f"Median: {df['power_kw'].median():.2f} kW")                              #Median der Ladeleistung berechnen
                                                                                #median(): Mittlerer Wert der sortierten Daten (50% liegen darunter bzw. darüber)
                                                                                #:.2f: Ergebnis auf zwei Nachkommastellen formatieren

print(f"Mode:   {df['power_kw'].mode()[0]:.2f} kW")                             #Modalwert der Ladeleistung berechnen
                                                                                #mode(): Gibt den/die am häufigsten vorkommenden Wert(e) als Series zurück
                                                                                #[0]: Ersten Modalwert aus der Series auswählen
                                                                                #:.2f: Ergebnis auf zwei Nachkommastellen formatieren

print()                                                                         #Leerzeile zur besseren Lesbarkeit

print('Note: mean (50.5 kW) >> median (22 kW) → strongly right-skewed.')        #Interpretation: Mittelwert deutlich grösser als Median → rechtsschiefe Verteilung
print('Most stations deliver 22 kW (AC), but a few fast-chargers (150–400 kW DC) pull the mean up.') #Wenige Ladestationen mit hoher Leistung erhöhen den Mittelwert

Mean:   50.50 kW
Median: 22.00 kW
Mode:   22.00 kW

Note: mean (50.5 kW) >> median (22 kW) → strongly right-skewed.
Most stations deliver 22 kW (AC), but a few fast-chargers (150–400 kW DC) pull the mean up.


### Task 3 — Quantiles for numerical columns

In [ ]:
# Task 3 – Quantiles of numerical columns

num_cols = ['power_kw', 'amperage', 'voltage']                              #Liste mit den numerischen Spalten erstellen, die untersucht werden sollen

df[num_cols].quantile([0.05, 0.25, 0.50, 0.75, 0.95]).round(1)              #Quantile der ausgewählten Spalten berechnen und auf eine Nachkommastelle runden
                                                                            #df_evcharging[num_cols]: Nur die in num_cols definierten Spalten auswählen
                                                                            #quantile(): Berechnet die angegebenen Quantile für jede ausgewählte Spalte
                                                                            #0.05: 5%-Quantil → 5% der Werte liegen darunter oder sind gleich gross
                                                                            #0.25: 25%-Quantil (Q1) → Unteres Quartil
                                                                            #0.50: 50%-Quantil (Q2) → Median
                                                                            #0.75: 75%-Quantil (Q3) → Oberes Quartil
                                                                            #0.95: 95%-Quantil → 95% der Werte liegen darunter oder sind gleich gross
                                                                            #round(1): Ergebnisse auf eine Nachkommastelle runden

,power_kw,amperage,voltage
0.05,11.0,16.0,230.0
0.25,22.0,32.0,230.0
0.50,22.0,32.0,230.0
0.75,22.0,50.0,230.0
0.95,225.0,563.0,400.0


### Task 4 — Variance and standard deviation

In [10]:
print('Sample variance:')                           #Überschrift für die Stichprobenvarianz ausgeben
print(df[num_cols].var().round(2))                  #Varianz für jede ausgewählte numerische Spalte berechnen
                                                    #df[num_cols]: Nur die zuvor definierten numerischen Spalten auswählen
                                                    #var(): Stichprobenvarianz berechnen (standardmässig ddof=1, Division durch n-1)
                                                    #round(2): Ergebnisse auf zwei Nachkommastellen runden
print()                                             #Leerzeile zur besseren Lesbarkeit

print('Sample standard deviation:')                 #Überschrift für die Stichprobenstandardabweichung ausgeben
print(df[num_cols].std().round(2))                  #Standardabweichung für jede ausgewählte numerische Spalte berechnen
                                                    #std(): Quadratwurzel der Stichprobenvarianz berechnen
                                                    #round(2): Ergebnisse auf zwei Nachkommastellen runden
print()                                             #Leerzeile zur besseren Lesbarkeit

print('The high std of power_kw (77.3 kW) reflects the wide mix of slow AC and fast DC chargers.') 
#Interpretation: Hohe Standardabweichung zeigt grosse Unterschiede zwischen den Ladeleistungen

Sample variance:
power_kw     5976.94
amperage    39833.08
voltage      5825.93
dtype: float64

Sample standard deviation:
power_kw     77.31
amperage    199.58
voltage      76.33
dtype: float64

The high std of power_kw (77.3 kW) reflects the wide mix of slow AC and fast DC chargers.


### Task 5 — Skewness and kurtosis of `power_kw`

In [13]:
skew = df['power_kw'].skew()                                    #Schiefe der Ladeleistung berechnen und in skew speichern
                                                                #skew(): Misst die Asymmetrie einer Verteilung
                                                                #Positiv: rechtsschief; negativ: linksschief; nahe 0: geringe Schiefe

kurt = df['power_kw'].kurtosis()                                #Exzess-Kurtosis der Ladeleistung berechnen und in kurt speichern
                                                                #kurtosis(): Misst insbesondere die Ausprägung der Verteilungsenden (Tails)
                                                                #Positiv: schwerere Tails; negativ: leichtere Tails als bei einer Normalverteilung
                                                                #Normalverteilung: Exzess-Kurtosis = 0

print(f'Skewness: {skew:.3f}')                                  #Schiefe mit drei Nachkommastellen ausgeben
                                                                #:.3f: Formatiert den Wert mit genau drei Nachkommastellen

print(f'Excess kurtosis: {kurt:.3f}')                           #Exzess-Kurtosis mit drei Nachkommastellen ausgeben

# Interpretation (fill in the blanks):
# Skewness of power_kw: 3.002 → the distribution is right-skewed because a few fast DC chargers (up to 400 kW) create a long tail to the right.
# Kurtosis of power_kw: 8.947 → the tails are heavier than a normal distribution.

#ERKENNTNIS: Skewness = 3.002 → Stark rechtsschiefe Verteilung, da wenige Schnellladestationen sehr hohe Ladeleistungen aufweisen
#ERKENNTNIS: Excess Kurtosis = 8.947 → Heavy-tailed (leptokurtische) Verteilung mit ausgeprägten Verteilungsenden und Extremwerten

Skewness: 3.002
Excess kurtosis: 8.947


### Task 6 — Contingency table: `power_type` × `plug_type`

In [14]:
ct = pd.crosstab(            #Kreuztabelle erstellen und in der Variable ct speichern
    df['power_type'],        #Zeilen: Verschiedene Ladearten (z.B. AC oder DC)
    df['plug_type'],         #Spalten: Verschiedene Steckertypen (z.B. CCS Combo 2, CHAdeMO, Type 2)
    margins=True,            #Zusätzliche Zeile und Spalte mit den Gesamtsummen hinzufügen
    margins_name='Total'     #Gesamtsummen mit 'Total' beschriften
)
ct                            #Kreuztabelle anzeigen
                              #Jede Zelle enthält die Anzahl Ladestationen mit der jeweiligen Kombination aus Ladeart und Steckertyp

print('Observation: DC stations exclusively use CCS Combo 2 or CHAdeMO plugs.')     
#Beobachtung: DC-Ladestationen verwenden im Datensatz ausschliesslich CCS Combo 2 oder CHAdeMO
print('AC 3-phase stations almost exclusively use Type 2 Outlet connectors.') 
#Beobachtung: Dreiphasige AC-Ladestationen verwenden fast ausschliesslich Type-2-Steckdosen
print('This reflects the technical standards: DC fast-charging uses CCS/CHAdeMO, AC uses Type 2.') 
#Interpretation: Diese Verteilung entspricht den typischen technischen Standards für DC- und AC-Laden

Observation: DC stations exclusively use CCS Combo 2 or CHAdeMO plugs.
AC 3-phase stations almost exclusively use Type 2 Outlet connectors.
This reflects the technical standards: DC fast-charging uses CCS/CHAdeMO, AC uses Type 2.


### Task 7 — Pearson correlation matrix

In [17]:
corr = df[num_cols].corr(method='pearson').round(3)             #Pearson-Korrelationsmatrix für die numerischen Spalten berechnen
                                                                #corr(): Berechnet die paarweisen Korrelationskoeffizienten
                                                                #method='pearson': Misst Stärke und Richtung linearer Zusammenhänge
                                                                #round(3): Auf drei Nachkommastellen runden
display(corr)                                                            #Korrelationsmatrix anzeigen

print('Interpretation:')                                                        #Interpretation der Korrelationsmatrix ausgeben
print('  power_kw ↔ amperage: strong positive correlation.')                    #Starker positiver Zusammenhang zwischen Ladeleistung und Stromstärke
print('  Higher charging power requires higher current (P = U × I).')           #Bei gleicher Spannung steigt die Leistung mit zunehmender Stromstärke
print()                                                                         #Leerzeile ausgeben
print('  power_kw ↔ voltage: moderate positive correlation.')                   #Moderater positiver Zusammenhang zwischen Ladeleistung und Spannung
print('  DC fast-chargers use higher voltages (400–800 V) than AC stations (230–400 V).')   #DC-Schnellladestationen arbeiten typischerweise mit höheren Spannungen
print()                                                                                     #Leerzeile ausgeben
print('  amperage ↔ voltage: weaker correlation — both contribute to power independently.') #Vergleichsweise schwächerer Zusammenhang zwischen Stromstärke und Spannung

# Interpretation:
# The strongest correlation is between power_kw and amperage (r = 0.997) #Stärkste Korrelation zwischen zwei unterschiedlichen Variablen
# This means: There is a very strong positive linear relationship — higher amperage is associated with higher charging power. #Höhere Stromstärke geht mit höherer Ladeleistung einher

#ERKENNTNIS: Pearson-Korrelation liegt zwischen -1 und +1; je näher an +1 oder -1, desto stärker der lineare Zusammenhang
#ACHTUNG: Korrelation bedeutet nicht automatisch Kausalität

,power_kw,amperage,voltage
power_kw,1.000,0.997,0.672
amperage,0.997,1.000,0.680
voltage,0.672,0.680,1.000


Interpretation:
  power_kw ↔ amperage: strong positive correlation.
  Higher charging power requires higher current (P = U × I).

  power_kw ↔ voltage: moderate positive correlation.
  DC fast-chargers use higher voltages (400–800 V) than AC stations (230–400 V).

  amperage ↔ voltage: weaker correlation — both contribute to power independently.


In [ ]:
# Interpretation:
# The strongest correlation is between ___ and ___ (r = ___)
# This means: ___


## Jupyter notebook --footer info--
(please always provide this at the end of each notebook)

In [ ]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('------------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('------------------------------------')